# Hierarchical Graph Representation with DiffPool on PROTEINS

Graph Classification on PROTEINS (TUDataset): Differentiable graph pooling learning hierarchical cluster assignment matrices. This notebook implements the approach with `DenseDiffPool` inside a `K3DiffPool` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `DenseDiffPool` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import TUDataset

title = "Hierarchical Graph Representation with DiffPool on PROTEINS"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = TUDataset(root="./data/PROTEINS", name="PROTEINS")
data = dataset[0]

# 2. DiffPool Architecture
class K3DiffPool(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels=2):
        super().__init__()
        self.conv1 = k3_layers.DenseGCNConv(in_channels, hidden_channels)
        self.pool1 = layers.Dense(10)  # cluster assignment matrix S
        self.conv2 = k3_layers.DenseGCNConv(hidden_channels, hidden_channels)
        self.lin = layers.Dense(out_channels)

    def call(self, x, adj):
        s = ops.softmax(self.pool1(x), axis=-1)
        x = ops.relu(self.conv1(x, adj))
        x, adj, link_loss, ent_loss = k3_layers.diff_pool(x, adj, s)
        x = ops.relu(self.conv2(x, adj))
        out = ops.mean(x, axis=1)
        return self.lin(out)

k3_model = K3DiffPool(dataset.num_features, 32, dataset.num_classes)

# 3. Forward Pass Verification
batch_size, num_nodes = 2, 20
dummy_x = keras.random.normal((batch_size, num_nodes, dataset.num_features))
dummy_adj = ops.ones((batch_size, num_nodes, num_nodes))

out = k3_model(dummy_x, dummy_adj)
print(f"DiffPool forward pass output shape: {out.shape} (Expected: ({batch_size}, {dataset.num_classes}))")

print("\n✓ K3-Node DiffPool execution completed successfully!")